# BiGRU — All 56 Features

Plain BiGRU (no CRF) with BCE + IoU loss.
All 56 features: 52 previous + 4 element-context (is_button, has_aria_hidden, in_table, in_details).

Data: `data/features.csv` (original dataset only).
80/20 page-level split (seed=42), 3-seed training.

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

DATA_DIR = Path("../data")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## 1. Load data

In [ ]:
all_df = pd.read_csv(DATA_DIR / "features.csv")
print(f"Loaded {len(all_df)} rows, {all_df['page_id'].nunique()} pages")

# Derive parent_tag_idx
TAG_MAP = {"body": 0, "header": 1, "nav": 2, "main": 3, "article": 4,
           "section": 5, "footer": 6, "aside": 7, "form": 8}
all_df["parent_tag_idx"] = all_df["parent_tag"].map(TAG_MAP).fillna(0).astype(int)

# Derive tag_transition and dist_to_heading per page
tag_trans = []
dist_head = []
for _, group in all_df.groupby("page_id", sort=False):
    transitions = (group["parent_tag"] != group["parent_tag"].shift(1)).astype(int)
    transitions.iloc[0] = 0
    tag_trans.append(transitions)
    last_h = -1
    dists = []
    for i, (_, row) in enumerate(group.iterrows()):
        if row["is_heading"] == 1:
            last_h = i
        dists.append(i - last_h if last_h >= 0 else i)
    dist_head.append(pd.Series(dists, index=group.index))

all_df["tag_transition"] = pd.concat(tag_trans)
all_df["dist_to_heading"] = pd.concat(dist_head)
print("Derived features added")

In [ ]:
# All 56 features
FEATURES = [
    # Positional (3)
    "position_pct", "total_lines", "depth",
    # Structural (9)
    "ancestor_depth_ratio", "parent_tag_idx",
    "in_header", "in_nav", "in_main", "in_article",
    "in_footer", "in_aside", "in_form",
    # Text content (8)
    "text_length", "word_count", "link_ratio",
    "is_link_only", "is_heading", "heading_level",
    "has_bold", "is_list_item",
    # Collapsed flags (2)
    "is_link_summary", "is_cookie_summary",
    # Text statistics (7)
    "punctuation_ratio", "sentence_count", "avg_word_length",
    "uppercase_ratio", "number_ratio", "type_token_ratio",
    "comma_count",
    # CSS class name patterns (3)
    "has_positive_class", "has_negative_class", "has_unlikely_class",
    # Element content flags (2)
    "has_image", "has_input",
    # Element context flags (4) — NEW
    "is_button", "has_aria_hidden", "in_table", "in_details",
    # Text uniqueness (5)
    "mean_idf", "max_idf", "line_frequency",
    "word_novelty", "word_novelty_sum",
    # Positional (1)
    "cumulative_text_pct",
    # Window context (2)
    "mean_text_length_w5", "mean_link_ratio_w5",
    # Block-level (5)
    "block_size", "block_text_density", "block_link_density",
    "relative_position_in_block", "sibling_text_variance",
    # Style-group (3)
    "style_group_size", "style_group_link_density", "style_group_mean_words",
    # Derived (2)
    "tag_transition", "dist_to_heading",
]

N_FEATURES = len(FEATURES)
assert N_FEATURES == 56, f"Expected 56, got {N_FEATURES}"
missing = [f for f in FEATURES if f not in all_df.columns]
assert not missing, f"Missing columns: {missing}"
print(f"Features: {N_FEATURES}")

In [4]:
# 80/20 split by page (seed=42)
all_pages = all_df["page_id"].unique()
rng = np.random.RandomState(42)
rng.shuffle(all_pages)
split_idx = int(len(all_pages) * 0.8)
train_ids = set(all_pages[:split_idx])
test_ids = set(all_pages[split_idx:])

train_df = all_df[all_df["page_id"].isin(train_ids)].reset_index(drop=True)
test_df = all_df[all_df["page_id"].isin(test_ids)].reset_index(drop=True)

# Normalize using train stats
feat_mean = train_df[FEATURES].values.astype(np.float32).mean(axis=0)
feat_std = train_df[FEATURES].values.astype(np.float32).std(axis=0)
feat_std[feat_std == 0] = 1.0

# Class imbalance weight
n_content = train_df["label"].sum()
n_total = len(train_df)
POS_WEIGHT = (n_total - n_content) / n_content

print(f"Total: {len(all_pages)} pages")
print(f"Train: {len(train_df)} rows, {len(train_ids)} pages")
print(f"Test:  {len(test_df)} rows, {len(test_ids)} pages")
print(f"Pos weight: {POS_WEIGHT:.3f}")

Total: 1738 pages
Train: 225756 rows, 1390 pages
Test:  53792 rows, 348 pages
Pos weight: 2.148


In [5]:
class PageDataset(Dataset):
    def __init__(self, pages):
        self.pages = pages
    def __len__(self):
        return len(self.pages)
    def __getitem__(self, idx):
        return self.pages[idx]["features"], self.pages[idx]["labels"]


def collate_fn(batch):
    feats_list, labels_list = zip(*batch)
    max_len = max(f.shape[0] for f in feats_list)
    n_feats = feats_list[0].shape[1]
    batch_size = len(batch)
    feats_padded = torch.zeros(batch_size, max_len, n_feats)
    labels_padded = torch.zeros(batch_size, max_len)
    mask = torch.zeros(batch_size, max_len, dtype=torch.bool)
    for i, (f, l) in enumerate(zip(feats_list, labels_list)):
        seq_len = f.shape[0]
        feats_padded[i, :seq_len] = f
        labels_padded[i, :seq_len] = l
        mask[i, :seq_len] = True
    return feats_padded, labels_padded, mask


def df_to_pages(df):
    pages = []
    for pid, group in df.groupby("page_id", sort=False):
        group = group.sort_values("line_num")
        feats = (group[FEATURES].values.astype(np.float32) - feat_mean) / feat_std
        labels = group["label"].values.astype(np.float32)
        pages.append({
            "page_id": pid,
            "features": torch.from_numpy(feats),
            "labels": torch.from_numpy(labels),
            "line_nums": group["line_num"].values,
            "span_lines": group["span_lines"].values,
        })
    return pages

train_pages = df_to_pages(train_df)
test_pages = df_to_pages(test_df)
print(f"Train: {len(train_pages)} pages, Test: {len(test_pages)} pages")
print(f"Avg lines/page: {np.mean([p['features'].shape[0] for p in train_pages]):.0f}")

Train: 1390 pages, Test: 348 pages
Avg lines/page: 162


## 2. Model: BiGRU (no CRF)

In [6]:
class BiGRUModel(nn.Module):
    def __init__(self, n_features, hidden=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.proj = nn.Linear(n_features, hidden)
        self.gru = nn.GRU(
            hidden, hidden // 2, num_layers=num_layers,
            batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        x = F.relu(self.proj(x))
        x, _ = self.gru(x)
        x = self.drop(x)
        return self.head(x).squeeze(2)


def compute_loss(logits, labels, mask):
    bce = F.binary_cross_entropy_with_logits(logits, labels, reduction="none")
    weights = torch.where(labels == 1, POS_WEIGHT, 1.0)
    bce_loss = (bce * weights * mask.float()).sum() / mask.sum()
    # Differentiable IoU loss
    probs = torch.sigmoid(logits)
    probs_m = probs * mask.float()
    labels_m = labels * mask.float()
    inter = (probs_m * labels_m).sum(dim=1)
    union = (probs_m + labels_m - probs_m * labels_m).sum(dim=1)
    iou = (inter + 1) / (union + 1)
    iou_loss = 1 - iou.mean()
    return bce_loss + 1.0 * iou_loss


print(f"BiGRU ready ({N_FEATURES} input features)")

BiGRU ready (52 input features)


In [7]:
def hard_iou(pred_bin, true_bin, line_nums, span_lines):
    pred_set, truth_set = set(), set()
    for j in range(len(pred_bin)):
        ln, span = int(line_nums[j]), int(span_lines[j])
        lines = set(range(ln, ln + span))
        if pred_bin[j]: pred_set |= lines
        if true_bin[j]: truth_set |= lines
    if not pred_set and not truth_set:
        return 1.0, 1.0, 1.0
    inter = len(pred_set & truth_set)
    union = len(pred_set | truth_set)
    iou = inter / union if union else 0.0
    prec = inter / len(pred_set) if pred_set else 0.0
    rec = inter / len(truth_set) if truth_set else 0.0
    return iou, prec, rec


def evaluate(model, pages, threshold=0.5):
    model.eval()
    results = []
    with torch.no_grad():
        for page in pages:
            x = page["features"].unsqueeze(0).to(DEVICE)
            logits = model(x)
            probs = torch.sigmoid(logits[0]).cpu().numpy()
            pred_bin = (probs >= threshold).astype(int)
            true_bin = page["labels"].numpy().astype(int)
            iou, prec, rec = hard_iou(pred_bin, true_bin,
                                       page["line_nums"], page["span_lines"])
            results.append({"page_id": page["page_id"], "iou": iou,
                            "precision": prec, "recall": rec})
    ious = [r["iou"] for r in results]
    return np.mean(ious), np.std(ious), results

## 3. Training

In [8]:
EPOCHS = 25
BATCH_SIZE = 16
LR = 5e-4
HIDDEN = 128
NUM_LAYERS = 2
DROPOUT = 0.2
SEEDS = [42, 123, 777]


def train_model(seed, epochs=EPOCHS, verbose=True):
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = BiGRUModel(N_FEATURES, hidden=HIDDEN, num_layers=NUM_LAYERS,
                       dropout=DROPOUT).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    train_loader = DataLoader(
        PageDataset(train_pages), batch_size=BATCH_SIZE,
        shuffle=True, collate_fn=collate_fn,
        generator=torch.Generator().manual_seed(seed),
    )

    best_iou = 0
    best_state = None

    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0
        n_batches = 0

        for feats, labels, mask in train_loader:
            feats, labels, mask = feats.to(DEVICE), labels.to(DEVICE), mask.to(DEVICE)
            logits = model(feats)
            loss = compute_loss(logits, labels, mask)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1

        scheduler.step()

        if epoch % 5 == 0 or epoch == 1:
            mi, ms, results = evaluate(model, test_pages)
            mp = np.mean([r["precision"] for r in results])
            mr = np.mean([r["recall"] for r in results])
            if mi > best_iou:
                best_iou = mi
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            if verbose:
                print(f"  ep {epoch:2d}  loss={epoch_loss/n_batches:.4f}  "
                      f"IoU={mi:.4f}+/-{ms:.4f}  P={mp:.3f} R={mr:.3f}  best={best_iou:.4f}")

    # Final eval
    mi, ms, _ = evaluate(model, test_pages)
    if mi > best_iou:
        best_iou = mi
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})
    return model, best_iou

In [ ]:
print("Training BiGRU (56 features)")
print("=" * 60)

results = []
best_overall_iou = 0
best_overall_model = None

for seed in SEEDS:
    print(f"\n--- seed={seed} ---")
    t0 = time.time()
    model, iou = train_model(seed)
    elapsed = time.time() - t0
    results.append({"seed": seed, "iou": iou, "time": elapsed})
    print(f"  Final: IoU={iou:.4f} ({elapsed:.0f}s)")
    if iou > best_overall_iou:
        best_overall_iou = iou
        best_overall_model = model

print(f"\n{'=' * 60}")
mean_iou = np.mean([r['iou'] for r in results])
std_iou = np.std([r['iou'] for r in results])
print(f"Mean IoU: {mean_iou:.4f} +/- {std_iou:.4f}")
print(f"Best IoU: {best_overall_iou:.4f}")

## 4. Analysis

In [10]:
mi, ms, page_results = evaluate(best_overall_model, test_pages)
ious = [r["iou"] for r in page_results]

print(f"Test IoU: {mi:.4f} +/- {ms:.4f}")
print(f"Precision: {np.mean([r['precision'] for r in page_results]):.4f}")
print(f"Recall:    {np.mean([r['recall'] for r in page_results]):.4f}")
print(f"Pages with IoU < 0.5: {sum(1 for x in ious if x < 0.5)}")
print(f"Pages with IoU = 1.0: {sum(1 for x in ious if x == 1.0)}")
print(f"Pages with IoU > 0.9: {sum(1 for x in ious if x > 0.9)}")
print()

ranked = sorted(page_results, key=lambda x: x["iou"])
print("Worst 10 pages:")
for r in ranked[:10]:
    print(f"  IoU={r['iou']:.3f}  P={r['precision']:.3f}  R={r['recall']:.3f}  page={r['page_id']}")

Test IoU: 0.9086 +/- 0.2007
Precision: 0.9267
Recall:    0.9709
Pages with IoU < 0.5: 19
Pages with IoU = 1.0: 200
Pages with IoU > 0.9: 275

Worst 10 pages:
  IoU=0.000  P=0.000  R=0.000  page=60
  IoU=0.000  P=0.000  R=0.000  page=156
  IoU=0.000  P=0.000  R=0.000  page=621
  IoU=0.053  P=0.053  R=1.000  page=915
  IoU=0.057  P=0.057  R=1.000  page=3284
  IoU=0.081  P=0.081  R=1.000  page=392
  IoU=0.103  P=1.000  R=0.103  page=897
  IoU=0.158  P=0.158  R=1.000  page=2273
  IoU=0.167  P=0.250  R=0.333  page=1542
  IoU=0.188  P=0.188  R=1.000  page=235


In [ ]:
# Permutation importance — which features matter most?
baseline_iou = mi
importances = {}

best_overall_model.eval()
for fi, feat_name in enumerate(FEATURES):
    shuffled_ious = []
    with torch.no_grad():
        for page in test_pages:
            x = page["features"].clone().unsqueeze(0).to(DEVICE)
            perm = torch.randperm(x.shape[1])
            x[0, :, fi] = x[0, perm, fi]
            logits = best_overall_model(x)
            probs = torch.sigmoid(logits[0]).cpu().numpy()
            pred_bin = (probs >= 0.5).astype(int)
            true_bin = page["labels"].numpy().astype(int)
            iou, _, _ = hard_iou(pred_bin, true_bin,
                                  page["line_nums"], page["span_lines"])
            shuffled_ious.append(iou)
    importances[feat_name] = baseline_iou - np.mean(shuffled_ious)

sorted_imp = sorted(importances.items(), key=lambda x: x[1], reverse=True)
print(f"{'Feature':<30s} {'IoU drop':>10s}")
print("-" * 42)
NEW_FEATURES = {"has_unlikely_class", "has_negative_class", "has_positive_class",
                "has_image", "has_input", "comma_count",
                "is_button", "has_aria_hidden", "in_table", "in_details"}
for name, drop in sorted_imp:
    marker = " <<<" if name in NEW_FEATURES else ""
    print(f"{name:<30s} {drop:>+10.4f}{marker}")

In [ ]:
# Save model
model_path = Path("../data/bigru_56f_model.pt")
torch.save({
    "model_state": best_overall_model.state_dict(),
    "config": {
        "n_features": N_FEATURES,
        "hidden": HIDDEN,
        "num_layers": NUM_LAYERS,
        "dropout": DROPOUT,
        "num_tags": 2,
    },
    "features": FEATURES,
    "feat_mean": feat_mean.tolist(),
    "feat_std": feat_std.tolist(),
    "test_iou": best_overall_iou,
}, model_path)
print(f"Saved to {model_path}")